# Topic 9: Trees (Binary Trees, BSTs, Heaps)

**Goal**: Master tree data structures — the foundation for most advanced DSA topics.  
**Time**: ~8-10 hours  
**Prereqs**: Topics 3-6

---

## Why Trees?

This is a **BIG** topic. Trees combine linked list concepts (nodes with pointers), recursion (most tree algorithms are recursive), and new traversal patterns.

Trees are everywhere:
- **File systems** — folders contain folders contain files
- **HTML DOM** — every webpage is a tree of elements
- **Org charts** — CEO → VPs → Directors → ...
- **Database indices** — B-trees power fast lookups
- **Heaps** — power priority queues
- Nearly every "Medium" and "Hard" interview problem involves trees

```
A BINARY TREE:

         1
       /   \
      2     3
     / \     \
    4   5     6

  • Each node has at most 2 children (left and right)
  • Node 1 is the ROOT (top of the tree)
  • Nodes 4, 5, 6 are LEAVES (no children)
  • The tree has DEPTH 2 (root is depth 0)
```

| Term | Meaning |
|---|---|
| Root | The topmost node (no parent) |
| Leaf | A node with no children |
| Depth | Distance from root to a node |
| Height | Distance from a node to deepest leaf |
| Subtree | A node and all its descendants |

---

## Part 1: Binary Tree Basics

### The TreeNode Class

Every tree is built from nodes. Each node holds a value and pointers to its left and right children:

```
  ┌──────────────┐
  │   val = 1    │
  │  ┌───┬───┐   │
  │  │ L │ R │   │
  └──┴─┬─┴─┬─┘───┘
       │   │
       ▼   ▼
      [2] [3]
```

### Building a Tree from a List

LeetCode represents trees as level-order lists: `[1, 2, 3, 4, 5, None, 6]`

```
 Index:  0   1   2   3   4    5    6
 List:  [1,  2,  3,  4,  5, None,  6]

         1          index 0
       /   \
      2     3       indices 1, 2
     / \     \
    4   5     6     indices 3, 4, (5=None), 6

 For node at index i:
   left child  = 2*i + 1
   right child = 2*i + 2
   parent      = (i - 1) // 2
```

In [1]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Build a binary tree from a level-order list (LeetCode style)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def print_tree(root, prefix="", is_left=True, is_root=True):
    """Pretty-print a binary tree with branch characters."""
    lines = []
    _build_lines(root, lines, "", True, True)
    print("\n".join(lines))


def _build_lines(node, lines, prefix, is_left, is_root):
    if node is None:
        return
    if is_root:
        connector = "    "
        new_prefix = "    "
    elif is_left:
        connector = prefix + " / "
        new_prefix = prefix + "    "
    else:
        connector = prefix + " \\ "
        new_prefix = prefix + "    "

    _build_lines(node.left, lines, "  " + prefix, True, False)
    lines.append(connector.rstrip() if is_root else connector.rstrip())
    lines.append(prefix + ("  " if is_root else "  ") + str(node.val))
    _build_lines(node.right, lines, "  " + prefix, False, False)


def display_tree(root):
    """Compact horizontal tree display."""
    lines = []
    _display(root, lines, "", "")
    return "\n".join(lines)


def _display(node, lines, prefix, child_prefix):
    if node is None:
        return
    lines.append(prefix + str(node.val))
    children = []
    if node.left:
        children.append((node.left, "L"))
    if node.right:
        children.append((node.right, "R"))
    for i, (child, side) in enumerate(children):
        if i < len(children) - 1:
            _display(child, lines, child_prefix + "├── ", child_prefix + "│   ")
        else:
            _display(child, lines, child_prefix + "└── ", child_prefix + "    ")


def simple_print(root):
    """Simple indented tree view."""
    def _print(node, level=0, label="Root"):
        if node:
            indent = "   " * level
            print(f"{indent}{label}: {node.val}")
            _print(node.left, level + 1, "L")
            _print(node.right, level + 1, "R")
    _print(root)


vals = [1, 2, 3, 4, 5, None, 6]
print(f"Building tree from {vals}:\n")

root = build_tree(vals)

print("    1")
print("   / \\")
print("  2   3")
print(" / \\   \\")
print("4   5   6")

print(f"\nRoot: {root.val}")
print(f"Root.left: {root.left.val}")
print(f"Root.right: {root.right.val}")
print(f"Root.left.left: {root.left.left.val}")

Building tree from [1, 2, 3, 4, 5, None, 6]:

    1
   / \
  2   3
 / \   \
4   5   6

Root: 1
Root.left: 2
Root.right: 3
Root.left.left: 4


In [2]:
print("Tree structure (horizontal view):\n")
print(display_tree(root))

print("\nIndented view:\n")
simple_print(root)

Tree structure (horizontal view):

1
├── 2
│   ├── 4
│   └── 5
└── 3
    └── 6

Indented view:

Root: 1
   L: 2
      L: 4
      R: 5
   R: 3
      R: 6


---

## Part 2: Tree Traversals — THE Core Topic

There are **four** ways to visit every node in a binary tree. Each visits the same nodes but in a different **order**.

Using this tree for ALL examples:

```
         1
       /   \
      2     3
     / \     \
    4   5     6


TRAVERSAL          RULE                        VISIT ORDER
─────────────────────────────────────────────────────────────
Inorder            Left → Root → Right          4, 2, 5, 1, 3, 6
Preorder           Root → Left → Right          1, 2, 4, 5, 3, 6
Postorder          Left → Right → Root          4, 5, 2, 6, 3, 1
Level-order (BFS)  Level by level, left→right   1, 2, 3, 4, 5, 6
```

### How to remember:

```
The prefix tells you WHEN to visit the ROOT:

  IN-order    →  root visited IN the middle    (L, ROOT, R)
  PRE-order   →  root visited BEFORE children  (ROOT, L, R)
  POST-order  →  root visited AFTER children   (L, R, ROOT)
```

### Step-by-step trace for INORDER:

```
         1
       /   \
      2     3
     / \     \
    4   5     6

 Call inorder(1)
   Call inorder(2)             ← go left
     Call inorder(4)           ← go left
       Call inorder(None) ✗   ← left is None, return
       VISIT 4 ✓              ← visit root
       Call inorder(None) ✗   ← right is None, return
     VISIT 2 ✓                ← visit root
     Call inorder(5)           ← go right
       Call inorder(None) ✗
       VISIT 5 ✓
       Call inorder(None) ✗
   VISIT 1 ✓                  ← visit root of whole tree
   Call inorder(3)             ← go right
     Call inorder(None) ✗
     VISIT 3 ✓
     Call inorder(6)
       Call inorder(None) ✗
       VISIT 6 ✓
       Call inorder(None) ✗

 Result: [4, 2, 5, 1, 3, 6]
```

In [3]:
def inorder_recursive(root):
    result = []
    def dfs(node):
        if not node:
            return
        dfs(node.left)
        result.append(node.val)
        print(f"  visit({node.val}) → {result}")
        dfs(node.right)
    dfs(root)
    return result


def inorder_iterative(root):
    result = []
    stack = []
    curr = root
    while curr or stack:
        while curr:
            print(f"  push {curr.val}, go left")
            stack.append(curr)
            curr = curr.left
        curr = stack.pop()
        result.append(curr.val)
        print(f"  left is None → pop {curr.val}, visit {curr.val} → go right")
        curr = curr.right
    return result


root = build_tree([1, 2, 3, 4, 5, None, 6])

print("=== INORDER (Left, Root, Right) ===\n")
print("Recursive:")
res = inorder_recursive(root)
print(f"\nResult: {res}")

print("\nIterative (using stack):")
res = inorder_iterative(root)
print(f"\nResult: {res}")

=== INORDER (Left, Root, Right) ===

Recursive:
  visit(4) → [4]
  visit(2) → [4, 2]
  visit(5) → [4, 2, 5]
  visit(1) → [4, 2, 5, 1]
  visit(3) → [4, 2, 5, 1, 3]
  visit(6) → [4, 2, 5, 1, 3, 6]

Result: [4, 2, 5, 1, 3, 6]

Iterative (using stack):
  push 1, go left
  push 2, go left
  push 4, go left
  left is None → pop 4, visit 4 → go right
  left is None → pop 2, visit 2 → go right
  push 5, go left
  left is None → pop 5, visit 5 → go right
  left is None → pop 1, visit 1 → go right
  push 3, go left
  left is None → pop 3, visit 3 → go right
  push 6, go left
  left is None → pop 6, visit 6 → go right

Result: [4, 2, 5, 1, 3, 6]


In [4]:
def preorder_recursive(root):
    result = []
    def dfs(node):
        if not node:
            return
        result.append(node.val)
        print(f"  visit({node.val}) → {result}")
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return result


def preorder_iterative(root):
    if not root:
        return []
    result = []
    stack = [root]
    while stack:
        node = stack.pop()
        result.append(node.val)
        pushes = []
        if node.right:
            stack.append(node.right)
            pushes.append(f"push right={node.right.val}")
        if node.left:
            stack.append(node.left)
            pushes.append(f"push left={node.left.val}")
        push_str = ", ".join(pushes) if pushes else "(leaf)"
        stack_vals = [n.val for n in stack]
        print(f"  pop {node.val} → visit {node.val}, {push_str} | stack={stack_vals}")
    return result


root = build_tree([1, 2, 3, 4, 5, None, 6])

print("=== PREORDER (Root, Left, Right) ===\n")
print("Recursive:")
res = preorder_recursive(root)
print(f"\nResult: {res}")

print("\nIterative (using stack):")
res = preorder_iterative(root)
print(f"\nResult: {res}")

=== PREORDER (Root, Left, Right) ===

Recursive:
  visit(1) → [1]
  visit(2) → [1, 2]
  visit(4) → [1, 2, 4]
  visit(5) → [1, 2, 4, 5]
  visit(3) → [1, 2, 4, 5, 3]
  visit(6) → [1, 2, 4, 5, 3, 6]

Result: [1, 2, 4, 5, 3, 6]

Iterative (using stack):
  pop 1 → visit 1, push right=3, push left=2 | stack=[3, 2]
  pop 2 → visit 2, push right=5, push left=4 | stack=[3, 5, 4]
  pop 4 → visit 4 (leaf) | stack=[3, 5]
  pop 5 → visit 5 (leaf) | stack=[3]
  pop 3 → visit 3, push right=6 | stack=[6]
  pop 6 → visit 6 (leaf) | stack=[]

Result: [1, 2, 4, 5, 3, 6]


In [5]:
def postorder_recursive(root):
    result = []
    def dfs(node):
        if not node:
            return
        dfs(node.left)
        dfs(node.right)
        result.append(node.val)
        print(f"  visit({node.val}) → {result}")
    dfs(root)
    return result


def postorder_iterative(root):
    """Two-stack approach: reverse of a modified preorder."""
    if not root:
        return []
    stack1 = [root]
    stack2 = []
    while stack1:
        node = stack1.pop()
        stack2.append(node.val)
        pushes = []
        if node.left:
            stack1.append(node.left)
            pushes.append(f"left={node.left.val}")
        if node.right:
            stack1.append(node.right)
            pushes.append(f"right={node.right.val}")
        push_str = f"push {' '.join(pushes)} to stack1" if pushes else "(leaf)"
        print(f"  Stack1 pop {node.val} → push to stack2, {push_str}")
    result = stack2[::-1]
    print(f"  Pop stack2 in order: {result}")
    return result


root = build_tree([1, 2, 3, 4, 5, None, 6])

print("=== POSTORDER (Left, Right, Root) ===\n")
print("Recursive:")
res = postorder_recursive(root)
print(f"\nResult: {res}")

print("\nIterative (two-stack method):")
res = postorder_iterative(root)
print(f"\nResult: {res}")

=== POSTORDER (Left, Right, Root) ===

Recursive:
  visit(4) → [4]
  visit(5) → [4, 5]
  visit(2) → [4, 5, 2]
  visit(6) → [4, 5, 2, 6]
  visit(3) → [4, 5, 2, 6, 3]
  visit(1) → [4, 5, 2, 6, 3, 1]

Result: [4, 5, 2, 6, 3, 1]

Iterative (two-stack method):
  Stack1 pop 1 → push to stack2, push left=2 right=3 to stack1
  Stack1 pop 3 → push to stack2, push right=6 to stack1
  Stack1 pop 6 → push to stack2 (leaf)
  Stack1 pop 2 → push to stack2, push left=4 right=5 to stack1
  Stack1 pop 5 → push to stack2 (leaf)
  Stack1 pop 4 → push to stack2 (leaf)
  Pop stack2 in order: [4, 5, 2, 6, 3, 1]

Result: [4, 5, 2, 6, 3, 1]


In [6]:
def level_order(root):
    if not root:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        result.append(node.val)
        children = []
        if node.left:
            queue.append(node.left)
            children.append(node.left.val)
        if node.right:
            queue.append(node.right)
            children.append(node.right.val)
        queue_vals = [n.val for n in queue]
        if children:
            print(f"  dequeue {node.val}, enqueue children {children} | queue={queue_vals}")
        else:
            print(f"  dequeue {node.val} (leaf) | queue={queue_vals}")
    return result


root = build_tree([1, 2, 3, 4, 5, None, 6])
print("=== LEVEL-ORDER / BFS ===\n")
print("         1")
print("       /   \\")
print("      2     3")
print("     / \\     \\")
print("    4   5     6\n")
res = level_order(root)
print(f"\nResult: {res}")

=== LEVEL-ORDER / BFS ===

         1
       /   \
      2     3
     / \     \
    4   5     6

  dequeue 1, enqueue children [2, 3] | queue=[2, 3]
  dequeue 2, enqueue children [4, 5] | queue=[3, 4, 5]
  dequeue 3, enqueue children [6] | queue=[4, 5, 6]
  dequeue 4 (leaf) | queue=[5, 6]
  dequeue 5 (leaf) | queue=[6]
  dequeue 6 (leaf) | queue=[]

Result: [1, 2, 3, 4, 5, 6]


---

## Part 3: Core Tree Problems

Now we apply traversals to solve real problems. Each problem follows the same pattern:
1. Understand the problem
2. Recognize which traversal/technique to use
3. Implement and trace

```
PROBLEM-SOLVING PATTERN FOR TREES:

  1. What info do I need from left subtree?
  2. What info do I need from right subtree?
  3. What do I do at the current node?
  4. What do I return to my parent?
```

---

### Problem 1: Maximum Depth / Height of Binary Tree (#104)

**Task**: Return the maximum depth (number of nodes on the longest root-to-leaf path).

```
         3
       /   \
      9    20
          /  \
        15    7

  Max depth = 3  (path: 3 → 20 → 15  or  3 → 20 → 7)
```

**Idea**: At each node, depth = `max(left_depth, right_depth) + 1`. Base case: `None → 0`.

In [7]:
def max_depth(root):
    if not root:
        return 0
    left = max_depth(root.left)
    right = max_depth(root.right)
    depth = max(left, right) + 1
    print(f"  node={root.val}:  left={left}, right={right} → depth={depth}")
    return depth


root = build_tree([3, 9, 20, None, None, 15, 7])
print("=== Maximum Depth ===\n")
print("         3")
print("       /   \\")
print("      9    20")
print("          /  \\")
print("        15    7\n")
result = max_depth(root)
print(f"\nMaximum depth: {result}")

=== Maximum Depth ===

         3
       /   \
      9    20
          /  \
        15    7

  node=9:  left=0, right=0 → depth=1
  node=15: left=0, right=0 → depth=1
  node=7:  left=0, right=0 → depth=1
  node=20: left=1, right=1 → depth=2
  node=3:  left=1, right=2 → depth=3

Maximum depth: 3


---

### Problem 2: Check if Balanced (#110)

**Task**: A tree is **balanced** if the left and right subtree heights differ by at most 1 at **every** node.

```
  BALANCED:              UNBALANCED:

       1                      1
      / \                    /
     2   3                  2
    / \                    /
   4   5                  3

  Heights differ by       Heights differ by
  at most 1 everywhere    2 at root (left=2, right=0)
```

**Idea**: DFS returns height if balanced, `-1` if any subtree is unbalanced. One pass — O(n).

In [8]:
def is_balanced(root):
    def check(node):
        if not node:
            return 0
        left_h = check(node.left)
        if left_h == -1:
            return -1
        right_h = check(node.right)
        if right_h == -1:
            return -1
        diff = abs(left_h - right_h)
        if diff > 1:
            print(f"  node={node.val}:  left_h={left_h}, right_h={right_h}, diff={diff} → UNBALANCED!")
            return -1
        height = max(left_h, right_h) + 1
        print(f"  node={node.val}:  left_h={left_h}, right_h={right_h}, diff={diff} → height={height}")
        return height

    return check(root) != -1


print("=== Check Balanced ===\n")

print("Test 1: Balanced tree [3, 9, 20, None, None, 15, 7]")
root1 = build_tree([3, 9, 20, None, None, 15, 7])
print(f"Result: {is_balanced(root1)}")

print("\nTest 2: Unbalanced tree [1, 2, None, 3, None, 4]")
root2 = build_tree([1, 2, None, 3, None, 4])
print(f"Result: {is_balanced(root2)}")

=== Check Balanced ===

Test 1: Balanced tree [3, 9, 20, None, None, 15, 7]
  node=9:  left_h=0, right_h=0, diff=0 → height=1
  node=15: left_h=0, right_h=0, diff=0 → height=1
  node=7:  left_h=0, right_h=0, diff=0 → height=1
  node=20: left_h=1, right_h=1, diff=0 → height=2
  node=3:  left_h=1, right_h=2, diff=1 → height=3
Result: True

Test 2: Unbalanced tree [1, 2, None, 3, None, 4]
  node=4:  left_h=0, right_h=0, diff=0 → height=1
  node=3:  left_h=1, right_h=0, diff=1 → height=2
  node=2:  left_h=2, right_h=0, diff=2 → UNBALANCED!
Result: False


---

### Problem 3: Diameter of Binary Tree (#543)

**Task**: Find the longest path between any two nodes. The path may NOT pass through the root.

```
         1
       /   \
      2     3
     / \
    4   5

  Longest path: 4 → 2 → 1 → 3  (length = 3 edges)
  Or equivalently: 5 → 2 → 1 → 3  (also 3 edges)

  At node 2: left_height=1 + right_height=1 = path through 2 = 2
  At node 1: left_height=2 + right_height=1 = path through 1 = 3 ← max!
```

**Idea**: At each node, the path through it = `left_height + right_height`. Track the global max.

In [9]:
def diameter_of_binary_tree(root):
    max_diameter = [0]

    def height(node):
        if not node:
            return 0
        left_h = height(node.left)
        right_h = height(node.right)
        path_through = left_h + right_h
        max_diameter[0] = max(max_diameter[0], path_through)
        print(f"  node={node.val}: left_h={left_h}, right_h={right_h}, "
              f"path_through={path_through}, max_so_far={max_diameter[0]}")
        return max(left_h, right_h) + 1

    height(root)
    return max_diameter[0]


root = build_tree([1, 2, 3, 4, 5])
print("=== Diameter of Binary Tree ===\n")
print("         1")
print("       /   \\")
print("      2     3")
print("     / \\")
print("    4   5\n")
print(f"\nDiameter: {diameter_of_binary_tree(root)}")

=== Diameter of Binary Tree ===

         1
       /   \
      2     3
     / \
    4   5

  node=4: left_h=0, right_h=0, path_through=0, max_so_far=0
  node=5: left_h=0, right_h=0, path_through=0, max_so_far=0
  node=2: left_h=1, right_h=1, path_through=2, max_so_far=2
  node=3: left_h=0, right_h=0, path_through=0, max_so_far=2
  node=1: left_h=2, right_h=1, path_through=3, max_so_far=3

Diameter: 3


---

### Problem 4: Same Tree (#100) & Symmetric Tree (#101)

**Same Tree**: Two trees are the same if they have identical structure and values.

```
    1       1
   / \     / \
  2   3   2   3     → Same? YES

    1       1
   / \     / \
  2   3   3   2     → Same? NO (left/right swapped)
```

**Symmetric Tree**: A tree is symmetric if its left subtree is a mirror of its right subtree.

```
       1
      / \
     2   2         → Symmetric? YES
    / \ / \
   3  4 4  3       (left subtree mirrors right subtree)
```

In [10]:
def is_same_tree(p, q):
    if not p and not q:
        print(f"  compare(None, None) → both None, True")
        return True
    if not p or not q:
        print(f"  compare({p.val if p else None}, {q.val if q else None}) → one is None, False")
        return False
    if p.val != q.val:
        print(f"  compare({p.val}, {q.val}) → values differ, False")
        return False
    print(f"  compare({p.val}, {q.val}) → values match, check children")
    return is_same_tree(p.left, q.left) and is_same_tree(p.right, q.right)


def is_symmetric(root):
    def is_mirror(left, right):
        if not left and not right:
            print(f"  mirror(None, None) → both None, True")
            return True
        if not left or not right:
            print(f"  mirror({left.val if left else None}, {right.val if right else None}) → one is None, False")
            return False
        if left.val != right.val:
            print(f"  mirror({left.val}, {right.val}) → values differ, False")
            return False
        print(f"  mirror({left.val}, {right.val}) → values match")
        return is_mirror(left.left, right.right) and is_mirror(left.right, right.left)

    if not root:
        return True
    return is_mirror(root.left, root.right)


print("=== Same Tree ===\n")
t1 = build_tree([1, 2, 3])
t2 = build_tree([1, 2, 3])
print(f"Tree [1,2,3] == [1,2,3]? {is_same_tree(t1, t2)}")

print("\n=== Symmetric Tree ===\n")
sym = build_tree([1, 2, 2, 3, 4, 4, 3])
print(f"[1,2,2,3,4,4,3] symmetric? {is_symmetric(sym)}")

print()
asym = build_tree([1, 2, 2, None, 3, None, 3])
print(f"[1,2,2,None,3,None,3] symmetric? {is_symmetric(asym)}")

=== Same Tree ===

  compare(1, 1) → values match, check children
  compare(2, 2) → values match, check children
  compare(None, None) → both None, True
  compare(None, None) → both None, True
  compare(3, 3) → values match, check children
  compare(None, None) → both None, True
  compare(None, None) → both None, True
Tree [1,2,3] == [1,2,3]? True

=== Symmetric Tree ===

  mirror(2, 2) → values match
  mirror(3, 3) → values match
  mirror(None, None) → both None, True
  mirror(None, None) → both None, True
  mirror(4, 4) → values match
  mirror(None, None) → both None, True
  mirror(None, None) → both None, True
[1,2,2,3,4,4,3] symmetric? True

  mirror(2, 2) → values match
  mirror(None, 3) → one is None, False
[1,2,2,None,3,None,3] symmetric? False


---

### Problem 5: Lowest Common Ancestor (LCA) (#236)

**Task**: Find the lowest node that is an ancestor of both `p` and `q`.

```
         3
       /   \
      5     1
     / \   / \
    6   2 0   8
       / \
      7   4

  LCA(5, 1) = 3     (3 is the root, both are in different subtrees)
  LCA(5, 4) = 5     (5 is an ancestor of 4, so 5 is the LCA)
  LCA(6, 4) = 5     (5 is the lowest common ancestor)
```

**Algorithm**:
```
  At each node:
    1. If node is None → return None
    2. If node IS p or q → return node (found a target!)
    3. Recurse left and right
    4. If BOTH left and right returned non-None → this node is the LCA
    5. Otherwise, return whichever side found something
```

In [11]:
def lowest_common_ancestor(root, p, q):
    if not root:
        return None
    if root.val == p or root.val == q:
        print(f"  at node {root.val}: FOUND target {root.val} → return {root.val}")
        return root

    left = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)

    if left and right:
        print(f"  at node {root.val}: left={left.val}, right={right.val} → BOTH sides found! LCA = {root.val}")
        return root
    if not left and not right:
        print(f"  at node {root.val}: left=None, right=None → return None")
        return None

    found = left if left else right
    side = "left" if left else "right"
    print(f"  at node {root.val}: left={left.val if left else None}, right={right.val if right else None} → return {found.val} (found on {side})")
    return found


root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
print("=== Lowest Common Ancestor ===\n")
print("         3")
print("       /   \\")
print("      5     1")
print("     / \\   / \\")
print("    6   2 0   8")
print("       / \\")
print("      7   4\n")

print("Finding LCA(5, 1):")
result = lowest_common_ancestor(root, 5, 1)
print(f"LCA(5, 1) = {result.val}")

print("\nFinding LCA(5, 4):")
result = lowest_common_ancestor(root, 5, 4)
print(f"LCA(5, 4) = {result.val}")

=== Lowest Common Ancestor ===

         3
       /   \
      5     1
     / \   / \
    6   2 0   8
       / \
      7   4

Finding LCA(5, 1):
  at node 6: left=None, right=None → return None
  at node 7: left=None, right=None → return None
  at node 4: left=None, right=None → return None
  at node 2: left=None, right=None → return None
  at node 5: FOUND target 5 → return 5
  at node 0: left=None, right=None → return None
  at node 8: left=None, right=None → return None
  at node 1: FOUND target 1 → return 1
  at node 3: left=5, right=1 → BOTH sides found! LCA = 3
LCA(5, 1) = 3

Finding LCA(5, 4):
  at node 6: left=None, right=None → return None
  at node 7: left=None, right=None → return None
  at node 4: FOUND target 4 → return 4
  at node 2: left=None, right=4 → return 4 (found on right)
  at node 5: FOUND target 5 → return 5
LCA(5, 4) = 5


---

### Problem 6: Binary Tree Level Order Traversal (#102)

**Task**: Return values level by level: `[[1], [2,3], [4,5,6]]`

```
         1          Level 0: [1]
       /   \
      2     3       Level 1: [2, 3]
     / \     \
    4   5     6     Level 2: [4, 5, 6]

  Output: [[1], [2, 3], [4, 5, 6]]
```

**Idea**: BFS with queue, but process nodes **level by level** — use `len(queue)` to know how many nodes are in the current level.

In [12]:
def level_order_levels(root):
    if not root:
        return []
    result = []
    queue = deque([root])
    level_num = 0

    while queue:
        level_size = len(queue)
        level_vals = [n.val for n in queue]
        print(f"  Level {level_num} ({level_size} node{'s' if level_size > 1 else ''}): processing {level_vals}")
        current_level = []

        for _ in range(level_size):
            node = queue.popleft()
            current_level.append(node.val)
            children = []
            if node.left:
                queue.append(node.left)
                children.append(str(node.left.val))
            if node.right:
                queue.append(node.right)
                children.append(str(node.right.val))
            if children:
                print(f"    node {node.val} → enqueue children: {', '.join(children)}")
            else:
                print(f"    node {node.val} (leaf)")

        print(f"    level result: {current_level}")
        result.append(current_level)
        level_num += 1

    return result


root = build_tree([1, 2, 3, 4, 5, None, 6])
print("=== Level Order Traversal ===\n")
print("         1")
print("       /   \\")
print("      2     3")
print("     / \\     \\")
print("    4   5     6\n")
result = level_order_levels(root)
print(f"\nResult: {result}")

=== Level Order Traversal ===

         1
       /   \
      2     3
     / \     \
    4   5     6

  Level 0 (1 node): processing [1]
    node 1 → enqueue children: 2, 3
    level result: [1]
  Level 1 (2 nodes): processing [2, 3]
    node 2 → enqueue children: 4, 5
    node 3 → enqueue children: 6
    level result: [2, 3]
  Level 2 (3 nodes): processing [4, 5, 6]
    node 4 (leaf)
    node 5 (leaf)
    node 6 (leaf)
    level result: [4, 5, 6]

Result: [[1], [2, 3], [4, 5, 6]]


---

### Problem 7: Binary Tree Maximum Path Sum (#124) — Hard

**Task**: Find the maximum path sum. A path can start and end at **any** node (not just root-to-leaf). Node values can be negative.

```
       -10
       /  \
      9   20
         /  \
        15   7

  Best path: 15 → 20 → 7 = 42
  (We skip -10 and 9 because they'd reduce the sum!)
```

**Algorithm**: At each node, compute the max gain from each side. The path through this node = `left_gain + node.val + right_gain`. But we can only extend ONE side upward (can't fork).

```
  At each node:
    left_gain  = max(0, max_gain(left))     ← take 0 if subtree is negative
    right_gain = max(0, max_gain(right))
    path_through_node = left_gain + val + right_gain   ← candidate for global max
    return val + max(left_gain, right_gain)            ← can only go ONE way up
```

In [13]:
def max_path_sum(root):
    global_max = [float('-inf')]

    def max_gain(node):
        if not node:
            return 0
        left_gain = max(0, max_gain(node.left))
        right_gain = max(0, max_gain(node.right))

        path_through = left_gain + node.val + right_gain
        global_max[0] = max(global_max[0], path_through)

        return_up = node.val + max(left_gain, right_gain)
        print(f"  node={node.val}: L_gain={left_gain}, R_gain={right_gain} | "
              f"path_through={path_through} | return_up={return_up} | global_max={global_max[0]}")
        return return_up

    max_gain(root)
    return global_max[0]


root = build_tree([-10, 9, 20, None, None, 15, 7])
print("=== Maximum Path Sum ===\n")
print("       -10")
print("       /  \\")
print("      9   20")
print("         /  \\")
print("        15   7\n")
print(f"\nMaximum path sum: {max_path_sum(root)}")

=== Maximum Path Sum ===

       -10
       /  \
      9   20
         /  \
        15   7

  node=9:  L_gain=0, R_gain=0 | path_through=9  | return_up=9  | global_max=9
  node=15: L_gain=0, R_gain=0 | path_through=15 | return_up=15 | global_max=15
  node=7:  L_gain=0, R_gain=0 | path_through=7  | return_up=7  | global_max=15
  node=20: L_gain=15, R_gain=7 | path_through=42 | return_up=35 | global_max=42
  node=-10: L_gain=9, R_gain=35 | path_through=34 | return_up=25 | global_max=42

Maximum path sum: 42


---

## Part 4: Binary Search Trees (BST)

A BST is a binary tree with the **search property**:

```
BST PROPERTY:

  For EVERY node:
    • All values in LEFT subtree  < node's value
    • All values in RIGHT subtree > node's value

         8
       /   \
      3     10
     / \      \
    1   6     14
       / \   /
      4   7 13

  Everything left of 8:  {1, 3, 4, 6, 7}  — all < 8  ✓
  Everything right of 8: {10, 13, 14}      — all > 8  ✓

  Inorder traversal: 1, 3, 4, 6, 7, 8, 10, 13, 14  ← SORTED!
```

**Key insight**: Inorder traversal of a BST always gives values in **sorted order**.

**Operations**:
| Operation | Average | Worst (skewed) |
|---|---|---|
| Search | O(log n) | O(n) |
| Insert | O(log n) | O(n) |
| Delete | O(log n) | O(n) |

In [14]:
def bst_search(root, target):
    node = root
    while node:
        if target == node.val:
            print(f"  at node {node.val}: FOUND!")
            return True
        elif target < node.val:
            print(f"  at node {node.val}: {target} < {node.val} → go LEFT")
            node = node.left
        else:
            print(f"  at node {node.val}: {target} > {node.val} → go RIGHT")
            node = node.right
    print(f"  hit None → NOT FOUND")
    return False


bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, None, None, 13])
print("=== BST Search ===\n")
print("         8")
print("       /   \\")
print("      3     10")
print("     / \\      \\")
print("    1   6     14")
print("       / \\   /")
print("      4   7 13\n")

print("Searching for 7:")
print(f"Found: {bst_search(bst, 7)}")

print("\nSearching for 5:")
print(f"Found: {bst_search(bst, 5)}")

=== BST Search ===

         8
       /   \
      3     10
     / \      \
    1   6     14
       / \   /
      4   7 13

Searching for 7:
  at node 8: 7 < 8 → go LEFT
  at node 3: 7 > 3 → go RIGHT
  at node 6: 7 > 6 → go RIGHT
  at node 7: FOUND!
Found: True

Searching for 5:
  at node 8: 5 < 8 → go LEFT
  at node 3: 5 > 3 → go RIGHT
  at node 6: 5 < 6 → go LEFT
  at node 4: 5 > 4 → go RIGHT
  hit None → NOT FOUND
Found: False


In [15]:
def bst_insert(root, val):
    if not root:
        return TreeNode(val)
    node = root
    path = []
    while node:
        if val < node.val:
            path.append(f"{val} < {node.val} → go left")
            if not node.left:
                node.left = TreeNode(val)
                path.append(f"place as left child of {node.val}")
                break
            node = node.left
        else:
            path.append(f"{val} > {node.val} → go right")
            if not node.right:
                node.right = TreeNode(val)
                path.append(f"place as right child of {node.val}")
                break
            node = node.right
    print(f"  insert({val}): {', '.join(path)}")
    return root


def inorder_list(root):
    if not root:
        return []
    return inorder_list(root.left) + [root.val] + inorder_list(root.right)


values = [8, 3, 10, 1, 6, 14, 4, 7, 13]
print("=== BST Insert ===\n")
print(f"Starting with empty BST, inserting: {values}\n")

bst_root = None
for v in values:
    if bst_root is None:
        bst_root = TreeNode(v)
        print(f"  insert({v}): tree is empty → {v} becomes root")
    else:
        bst_root = bst_insert(bst_root, v)

print(f"\nFinal BST (inorder = sorted): {inorder_list(bst_root)}")
print(f"\nTree structure:")
print(display_tree(bst_root))

=== BST Insert ===

Starting with empty BST, inserting: [8, 3, 10, 1, 6, 14, 4, 7, 13]

  insert(8): tree is empty → 8 becomes root
  insert(3): 3 < 8 → go left, place as left child of 8
  insert(10): 10 > 8 → go right, place as right child of 8
  insert(1): 1 < 8 → go left, 1 < 3 → go left, place as left child of 3
  insert(6): 6 < 8 → go left, 6 > 3 → go right, place as right child of 3
  insert(14): 14 > 8 → go right, 14 > 10 → go right, place as right child of 10
  insert(4): 4 < 8 → go left, 4 > 3 → go right, 4 < 6 → go left, place as left child of 6
  insert(7): 7 < 8 → go left, 7 > 3 → go right, 7 > 6 → go right, place as right child of 6
  insert(13): 13 > 8 → go right, 13 > 10 → go right, 13 < 14 → go left, place as left child of 14

Final BST (inorder = sorted): [1, 3, 4, 6, 7, 8, 10, 13, 14]

Tree structure:
8
├── 3
│   ├── 1
│   └── 6
│       ├── 4
│       └── 7
└── 10
    └── 14
        └── 13


---

### BST Deletion — The Tricky One

Three cases when deleting a node:

```
CASE 1: Leaf node (no children)
  → Just remove it

  Delete 4:         8              8
                   / \            / \
                  3   10   →     3   10
                 / \            /
                1   4          1

CASE 2: One child
  → Replace node with its child

  Delete 10:        8              8
                   / \            / \
                  3   10   →     3   14
                       \
                       14

CASE 3: Two children
  → Find INORDER SUCCESSOR (smallest in right subtree)
  → Copy its value to current node
  → Delete the successor

  Delete 3:         8              8
                   / \            / \
                  3   10   →     4   10
                 / \            /
                1   6         1   6
                   /               
                  4          (successor was 4, smallest in right subtree)
```

In [16]:
def bst_delete(root, key):
    if not root:
        return None

    if key < root.val:
        print(f"  at {root.val}: {key} < {root.val} → go left")
        root.left = bst_delete(root.left, key)
    elif key > root.val:
        print(f"  at {root.val}: {key} > {root.val} → go right")
        root.right = bst_delete(root.right, key)
    else:
        if not root.left and not root.right:
            print(f"  FOUND {key} — Case 1: leaf → remove")
            return None
        elif not root.left or not root.right:
            print(f"  FOUND {key} — Case 2: one child → replace with child")
            return root.left if root.left else root.right
        else:
            print(f"  FOUND {key} — Case 3: two children → find inorder successor")
            successor = root.right
            while successor.left:
                successor = successor.left
            print(f"  Inorder successor: {successor.val} (smallest in right subtree)")
            print(f"  Replace {root.val} with {successor.val}, delete {successor.val} from right subtree")
            root.val = successor.val
            root.right = bst_delete(root.right, successor.val)
    return root


bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, None, None, 13])
print("=== BST Delete ===\n")
print(f"Starting BST (inorder): {inorder_list(bst)}")

print("\nDelete 4 (leaf):")
bst = bst_delete(bst, 4)
print(f"  After: {inorder_list(bst)}")

print("\nDelete 10 (one child):")
bst = bst_delete(bst, 10)
print(f"  After: {inorder_list(bst)}")

print("\nDelete 3 (two children):")
bst = bst_delete(bst, 3)
print(f"  After: {inorder_list(bst)}")

=== BST Delete ===

Starting BST (inorder): [1, 3, 4, 6, 7, 8, 10, 13, 14]

Delete 4 (leaf):
  at 8: 4 < 8 → go left
  at 3: 4 > 3 → go right
  at 6: 4 < 6 → go left
  FOUND 4 — Case 1: leaf → remove
  After: [1, 3, 6, 7, 8, 10, 13, 14]

Delete 10 (one child):
  at 8: 10 > 8 → go right
  FOUND 10 — Case 2: one child → replace with child
  After: [1, 3, 6, 7, 8, 13, 14]

Delete 3 (two children):
  at 8: 3 < 8 → go left
  FOUND 3 — Case 3: two children → find inorder successor
  Inorder successor: 6 (smallest in right subtree)
  Replace 3 with 6, delete 6 from right subtree
  After: [1, 6, 7, 8, 13, 14]


---

### Validate BST (#98)

**Task**: Check if a binary tree is a valid BST.

```
  WRONG approach:  just check node.left.val < node.val < node.right.val
  This FAILS because it doesn't check the ENTIRE subtree:

       5
      / \
     1   6       ← Looks valid at each node locally...
        / \
       3   7     ← But 3 < 5, so it's in the WRONG subtree!

  CORRECT approach: pass valid range (min, max) down the tree
```

In [17]:
def is_valid_bst(root):
    def validate(node, lo, hi):
        if not node:
            return True
        if not (lo < node.val < hi):
            print(f"  node={node.val}: val={node.val}, range=({lo}, {hi}) ✗ INVALID! {node.val} not in ({lo}, {hi})")
            return False
        print(f"  node={node.val}: val={node.val}, range=({lo}, {hi}) ✓")
        return validate(node.left, lo, node.val) and validate(node.right, node.val, hi)

    return validate(root, float('-inf'), float('inf'))


print("=== Validate BST ===\n")

print("Test 1: Valid BST [8, 3, 10, 1, 6]")
valid = build_tree([8, 3, 10, 1, 6])
print(f"Valid BST? {is_valid_bst(valid)}")

print("\nTest 2: Invalid BST [5, 1, 6, None, None, 3, 7]")
invalid = build_tree([5, 1, 6, None, None, 3, 7])
print(f"Valid BST? {is_valid_bst(invalid)}")

=== Validate BST ===

Test 1: Valid BST [8, 3, 10, 1, 6]
  node=8: val=8, range=(-inf, inf) ✓
  node=3: val=3, range=(-inf, 8) ✓
  node=1: val=1, range=(-inf, 3) ✓
  node=6: val=6, range=(3, 8) ✓
  node=10: val=10, range=(8, inf) ✓
Valid BST? True

Test 2: Invalid BST [5, 1, 6, None, None, 3, 7]
  node=5: val=5, range=(-inf, inf) ✓
  node=1: val=1, range=(-inf, 5) ✓
  node=6: val=6, range=(5, inf) ✓
  node=3: val=3, range=(5, 6) ✗ INVALID! 3 not in (5, 6)
Valid BST? False


In [18]:
def kth_smallest(root, k):
    count = [0]
    result = [None]

    def inorder(node):
        if not node or result[0] is not None:
            return
        inorder(node.left)
        count[0] += 1
        if count[0] == k:
            print(f"  inorder visit #{count[0]}: node={node.val} ← FOUND! k={k}")
            result[0] = node.val
            return
        print(f"  inorder visit #{count[0]}: node={node.val}")
        inorder(node.right)

    inorder(root)
    return result[0]


bst = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7])
print("=== Kth Smallest in BST (#230) ===\n")
print("         8")
print("       /   \\")
print("      3     10")
print("     / \\      \\")
print("    1   6     14")
print("       / \\")
print("      4   7\n")
print("Inorder gives sorted order — just stop at kth element!\n")

for k in [1, 3, 5]:
    print(f"Finding k={k}{' (smallest)' if k == 1 else ''}:")
    val = kth_smallest(bst, k)
    print(f"  {k}{'st' if k == 1 else 'rd' if k == 3 else 'th'} smallest = {val}\n")

=== Kth Smallest in BST (#230) ===

         8
       /   \
      3     10
     / \      \
    1   6     14
       / \
      4   7

Inorder gives sorted order — just stop at kth element!

Finding k=1 (smallest):
  inorder visit #1: node=1 ← FOUND! k=1
  1st smallest = 1

Finding k=3:
  inorder visit #1: node=1
  inorder visit #2: node=3
  inorder visit #3: node=4 ← FOUND! k=3
  3rd smallest = 4

Finding k=5:
  inorder visit #1: node=1
  inorder visit #2: node=3
  inorder visit #3: node=4
  inorder visit #4: node=6
  inorder visit #5: node=7 ← FOUND! k=5
  5th smallest = 7


In [19]:
def sorted_array_to_bst(nums):
    if not nums:
        return None
    mid = len(nums) // 2
    node = TreeNode(nums[mid])
    if len(nums) == 1:
        print(f"  build({nums}) → mid={nums[mid]} (leaf)")
    else:
        print(f"  build({nums}) → mid={nums[mid]}")
    node.left = sorted_array_to_bst(nums[:mid])
    node.right = sorted_array_to_bst(nums[mid + 1:])
    return node


nums = [-10, -3, 0, 5, 9]
print("=== Sorted Array to BST (#108) ===\n")
print(f"Input: {nums}")
print("\nPick middle as root, recurse on left and right halves:\n")

bst = sorted_array_to_bst(nums)
print(f"\nResult:")
print(display_tree(bst))

print(f"\nInorder (should be sorted): {inorder_list(bst)}")

def silent_validate(node, lo, hi):
    if not node:
        return True
    if not (lo < node.val < hi):
        return False
    return silent_validate(node.left, lo, node.val) and silent_validate(node.right, node.val, hi)

print(f"Is valid BST? {silent_validate(bst, float('-inf'), float('inf'))}")

=== Sorted Array to BST (#108) ===

Input: [-10, -3, 0, 5, 9]

Pick middle as root, recurse on left and right halves:

  build([-10, -3, 0, 5, 9]) → mid=0
  build([-10, -3]) → mid=-10
  build([-3]) → mid=-3 (leaf)
  build([5, 9]) → mid=5
  build([9]) → mid=9 (leaf)

Result:
0
├── -10
│   └── -3
└── 5
    └── 9

Inorder (should be sorted): [-10, -3, 0, 5, 9]
Is valid BST? True


---

## Part 5: Heaps / Priority Queues

A **heap** is a complete binary tree stored as an **array** where:
- **Min-heap**: parent ≤ children (root is the minimum)
- **Max-heap**: parent ≥ children (root is the maximum)

```
MIN-HEAP EXAMPLE:

  Array:  [1, 3, 5, 7, 9, 8]
  Index:   0  1  2  3  4  5

           1             parent(i)     = (i - 1) // 2
         /   \           left_child(i) = 2 * i + 1
        3     5          right_child(i)= 2 * i + 2
       / \   /
      7   9 8

  Every parent ≤ its children:
    1 ≤ 3, 5    ✓
    3 ≤ 7, 9    ✓
    5 ≤ 8       ✓


HEAP vs BST:
  BST:  left < root < right  (fully sorted structure)
  Heap: parent ≤ children    (only parent-child ordering)

  Heap gives O(1) access to min/max, BST gives O(log n) search.
```

**Key operations**:
| Operation | Time |
|---|---|
| Get min/max | O(1) |
| Insert (push) | O(log n) |
| Extract min/max (pop) | O(log n) |
| Build heap from array | O(n) |

In [20]:
class MinHeap:
    def __init__(self):
        self.data = []

    def push(self, val):
        self.data.append(val)
        self._heapify_up(len(self.data) - 1)

    def pop(self):
        if len(self.data) == 1:
            return self.data.pop()
        root_val = self.data[0]
        self.data[0] = self.data.pop()
        self._heapify_down(0)
        return root_val

    def peek(self):
        return self.data[0] if self.data else None

    def _heapify_up(self, idx):
        swaps = []
        while idx > 0:
            parent = (idx - 1) // 2
            if self.data[idx] < self.data[parent]:
                swaps.append(f"swap {self.data[idx]}↔{self.data[parent]}")
                self.data[idx], self.data[parent] = self.data[parent], self.data[idx]
                idx = parent
            else:
                break
        return swaps

    def _heapify_down(self, idx):
        swaps = []
        n = len(self.data)
        while True:
            smallest = idx
            left = 2 * idx + 1
            right = 2 * idx + 2
            if left < n and self.data[left] < self.data[smallest]:
                smallest = left
            if right < n and self.data[right] < self.data[smallest]:
                smallest = right
            if smallest != idx:
                swaps.append(f"swap {self.data[idx]}↔{self.data[smallest]}")
                self.data[idx], self.data[smallest] = self.data[smallest], self.data[idx]
                idx = smallest
            else:
                break
        return swaps


print("=== Min-Heap from Scratch ===\n")
values = [7, 3, 9, 1, 5, 8]
print(f"Pushing values: {values}\n")

heap = MinHeap()
for v in values:
    heap.data.append(v)
    swaps = heap._heapify_up(len(heap.data) - 1)
    if swaps:
        print(f"  push({v}): insert at end → {heap.data[:len(heap.data)]}, bubble up: {' → '.join(swaps)} → {list(heap.data)}")
    else:
        print(f"  push({v}): insert at end → {list(heap.data)}, no swap needed")

print(f"\nFinal heap: {list(heap.data)}")
print(f"\n           {heap.data[0]}")
print(f"         /   \\")
print(f"        {heap.data[1]}     {heap.data[2]}")
print(f"       / \\   /")
print(f"      {heap.data[3]}   {heap.data[4]} {heap.data[5]}")

print("\nPopping all (should come out sorted):")
sorted_vals = []
while heap.data:
    if len(heap.data) == 1:
        val = heap.data.pop()
        print(f"  pop → {val}: last element")
    else:
        val = heap.data[0]
        heap.data[0] = heap.data.pop()
        state_after_swap = list(heap.data)
        swaps = heap._heapify_down(0)
        if swaps:
            print(f"  pop → {val}: swap root↔last {state_after_swap}, bubble down: {' → '.join(swaps)} → {list(heap.data)}")
        else:
            print(f"  pop → {val}: swap root↔last {state_after_swap}, no children to bubble")
    sorted_vals.append(val)

print(f"\nPopped in order: {sorted_vals} ← sorted!")

=== Min-Heap from Scratch ===

Pushing values: [7, 3, 9, 1, 5, 8]

  push(7): heap = [7]
  push(3): insert at end → [7, 3], bubble up: swap 3↔7 → [3, 7]
  push(9): insert at end → [3, 7, 9], no swap needed
  push(1): insert at end → [3, 7, 9, 1], bubble up: swap 1↔7 → swap 1↔3 → [1, 3, 9, 7]
  push(5): insert at end → [1, 3, 9, 7, 5], no swap needed
  push(8): insert at end → [1, 3, 9, 7, 5, 8], no swap needed

Final heap: [1, 3, 9, 7, 5, 8]

           1
         /   \
        3     9
       / \   /
      7   5 8

Popping all (should come out sorted):
  pop → 1: swap root↔last [8, 3, 9, 7, 5], bubble down: swap 8↔3 → swap 8↔5 → [3, 5, 9, 7, 8]
  pop → 3: swap root↔last [8, 5, 9, 7], bubble down: swap 8↔5 → swap 8↔7 → [5, 7, 9, 8]
  pop → 5: swap root↔last [8, 7, 9], bubble down: swap 8↔7 → [7, 8, 9]
  pop → 7: swap root↔last [9, 8], bubble down: swap 9↔8 → [8, 9]
  pop → 8: swap root↔last [9], no children to bubble
  pop → 9: last element

Popped in order: [1, 3, 5, 7, 8, 9] ← sorted!

In [21]:
import heapq

print("=== Python's heapq Module ===\n")
print("heapq provides a MIN-heap (smallest element first).")
print("For a MAX-heap, negate the values.\n")

h = []
print("Push: 7, 3, 9, 1, 5")
for v in [7, 3, 9, 1, 5]:
    heapq.heappush(h, v)
    print(f"  after push({v}): {h}")

print(f"\nPeek (smallest): {h[0]}")

print("\nPop all:")
while h:
    val = heapq.heappop(h)
    print(f"  pop → {val} | remaining: {h}")

print("\n--- heapq shortcuts ---")
data = [7, 3, 9, 1, 5, 8, 2]
print(f"3 smallest from {data}: {heapq.nsmallest(3, data)}")
print(f"3 largest  from {data}: {heapq.nlargest(3, data)}")

=== Python's heapq Module ===

heapq provides a MIN-heap (smallest element first).
For a MAX-heap, negate the values.

Push: 7, 3, 9, 1, 5
  after push(7): [7]
  after push(3): [3, 7]
  after push(9): [3, 7, 9]
  after push(1): [1, 3, 9, 7]
  after push(5): [1, 3, 9, 7, 5]

Peek (smallest): 1

Pop all:
  pop → 1 | remaining: [3, 5, 9, 7]
  pop → 3 | remaining: [5, 7, 9]
  pop → 5 | remaining: [7, 9]
  pop → 7 | remaining: [9]
  pop → 9 | remaining: []

--- heapq shortcuts ---
3 smallest from [7, 3, 9, 1, 5, 8, 2]: [1, 2, 3]
3 largest  from [7, 3, 9, 1, 5, 8, 2]: [9, 8, 7]


---

### Heap Problems

The classic pattern: **"Top K" or "Kth largest/smallest"** → use a heap.

```
KTH LARGEST TRICK:

  Use a MIN-HEAP of size k.
  The root of the heap = the kth largest element.

  Why? The heap holds the k largest elements seen so far.
  The smallest of those k elements (heap root) = kth largest overall.

  nums = [3, 2, 1, 5, 6, 4],  k = 2

  Process 3: heap = [3]          (size < k, just add)
  Process 2: heap = [2, 3]       (size = k)
  Process 1: 1 < heap[0]=2       (skip, too small)
  Process 5: 5 > heap[0]=2       (pop 2, push 5) → heap = [3, 5]
  Process 6: 6 > heap[0]=3       (pop 3, push 6) → heap = [5, 6]
  Process 4: 4 < heap[0]=5       (skip)

  Answer: heap[0] = 5  (2nd largest)
```

In [22]:
def find_kth_largest(nums, k):
    heap = []
    for num in nums:
        if len(heap) < k:
            heapq.heappush(heap, num)
            print(f"  process {num}: heap size < k → push → heap={sorted(heap)}")
        elif num > heap[0]:
            old = heapq.heapreplace(heap, num)
            print(f"  process {num}: {num} > heap_min={old} → pop {old}, push {num} → heap={sorted(heap)}")
        else:
            print(f"  process {num}: {num} ≤ heap_min={heap[0]} → skip")
    return heap[0]


nums = [3, 2, 1, 5, 6, 4]
k = 2
print(f"=== Kth Largest Element (#215) ===\n")
print(f"nums = {nums}, k = {k}\n")
result = find_kth_largest(nums, k)
print(f"\n{k}nd largest = {result}")

=== Kth Largest Element (#215) ===

nums = [3, 2, 1, 5, 6, 4], k = 2

  process 3: heap size < k → push → heap=[3]
  process 2: heap size < k → push → heap=[2, 3]
  process 1: 1 ≤ heap_min=2 → skip
  process 5: 5 > heap_min=2 → pop 2, push 5 → heap=[3, 5]
  process 6: 6 > heap_min=3 → pop 3, push 6 → heap=[5, 6]
  process 4: 4 ≤ heap_min=5 → skip

2nd largest = 5


In [23]:
from collections import Counter


def top_k_frequent(nums, k):
    freq = Counter(nums)
    print(f"Step 1 — Count frequencies:")
    print(f"  {dict(freq)}")

    print(f"\nStep 2 — Use min-heap of size k on (frequency, value):")
    heap = []
    for val, count in freq.items():
        if len(heap) < k:
            heapq.heappush(heap, (count, val))
            print(f"  process ({count}, {val}): heap size < k → push → heap={sorted(heap, reverse=True)}")
        elif count > heap[0][0]:
            old = heapq.heapreplace(heap, (count, val))
            print(f"  process ({count}, {val}): freq {count} > heap_min_freq {old[0]} → replace → heap={sorted(heap, reverse=True)}")
        else:
            print(f"  process ({count}, {val}): freq {count} ≤ heap_min_freq {heap[0][0]} → skip")

    return [val for _, val in heap]


nums = [1, 1, 1, 2, 2, 3]
k = 2
print(f"=== Top K Frequent Elements (#347) ===\n")
print(f"nums = {nums}, k = {k}\n")
result = top_k_frequent(nums, k)
print(f"\nTop {k} frequent elements: {result}")

=== Top K Frequent Elements (#347) ===

nums = [1, 1, 1, 2, 2, 3], k = 2

Step 1 — Count frequencies:
  {1: 3, 2: 2, 3: 1}

Step 2 — Use min-heap of size k on (frequency, value):
  process (3, 1): heap size < k → push → heap=[(3, 1)]
  process (2, 2): heap size < k → push → heap=[(2, 2), (3, 1)]
  process (1, 3): freq 1 ≤ heap_min_freq 2 → skip

Top 2 frequent elements: [1, 2]


In [24]:
def merge_k_sorted(lists):
    heap = []
    for i, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst[0], i, 0))

    print(f"  heap: {heap}")
    result = []

    while heap:
        val, list_idx, elem_idx = heapq.heappop(heap)
        result.append(val)
        next_idx = elem_idx + 1
        if next_idx < len(lists[list_idx]):
            next_val = lists[list_idx][next_idx]
            heapq.heappush(heap, (next_val, list_idx, next_idx))
            print(f"  pop (val={val}, list={list_idx}) → result={result} | push next from list {list_idx}: val={next_val}")
        else:
            print(f"  pop (val={val}, list={list_idx}) → result={result} | list {list_idx} exhausted")

    return result


lists = [[1, 4, 5], [1, 3, 4], [2, 6]]
print("=== Merge K Sorted Lists (#23) ===\n")
print("Lists:")
for lst in lists:
    print(f"  {lst}")
print("\nUsing min-heap to always pick the smallest available element:\n")
result = merge_k_sorted(lists)
print(f"\nMerged: {result}")

=== Merge K Sorted Lists (#23) ===

Lists:
  [1, 4, 5]
  [1, 3, 4]
  [2, 6]

Using min-heap to always pick the smallest available element:

  heap: [(1, 0, 0), (1, 1, 0), (2, 2, 0)]
  pop (val=1, list=0) → result=[1] | push next from list 0: val=4
  pop (val=1, list=1) → result=[1, 1] | push next from list 1: val=3
  pop (val=2, list=2) → result=[1, 1, 2] | push next from list 2: val=6
  pop (val=3, list=1) → result=[1, 1, 2, 3] | push next from list 1: val=4
  pop (val=4, list=0) → result=[1, 1, 2, 3, 4] | push next from list 0: val=5
  pop (val=4, list=1) → result=[1, 1, 2, 3, 4, 4] | list 1 exhausted
  pop (val=5, list=0) → result=[1, 1, 2, 3, 4, 4, 5] | list 0 exhausted
  pop (val=6, list=2) → result=[1, 1, 2, 3, 4, 4, 5, 6] | list 2 exhausted

Merged: [1, 1, 2, 3, 4, 4, 5, 6]


---

## Part 6: Serialize and Deserialize Binary Tree (#297)

**Task**: Convert a tree to a string and back. Must handle any binary tree, including `None` nodes.

**Strategy**: Use **preorder traversal** with `"#"` markers for `None`:

```
         1
       /   \
      2     3
           / \
          4   5

  Preorder with None markers:
    visit 1, visit 2, mark #, mark #, visit 3, visit 4, mark #, mark #, visit 5, mark #, mark #

  Serialized: "1,2,#,#,3,4,#,#,5,#,#"

  Deserialize: read values left-to-right, recursively build left then right.
```

In [25]:
def serialize(root):
    tokens = []
    def preorder(node):
        if not node:
            tokens.append("#")
            print(f"  null    → {tokens}")
            return
        tokens.append(str(node.val))
        print(f"  visit {node.val} → {tokens}")
        preorder(node.left)
        preorder(node.right)
    preorder(root)
    return ",".join(tokens)


def deserialize(data):
    tokens = iter(data.split(","))
    def build():
        val = next(tokens)
        if val == "#":
            print(f"  read '#' → None")
            return None
        print(f"  read '{val}' → create node({val})")
        node = TreeNode(int(val))
        node.left = build()
        node.right = build()
        return node
    return build()


root = build_tree([1, 2, 3, None, None, 4, 5])
print("=== Serialize / Deserialize (#297) ===\n")
print("Original tree:")
print(display_tree(root))

print("\n--- Serializing ---")
encoded = serialize(root)
print(f"\nSerialized: \"{encoded}\"")

print("\n--- Deserializing ---")
decoded = deserialize(encoded)
print(f"\nReconstructed tree:")
print(display_tree(decoded))

def trees_equal(a, b):
    if not a and not b:
        return True
    if not a or not b:
        return False
    return a.val == b.val and trees_equal(a.left, b.left) and trees_equal(a.right, b.right)

print(f"\nTrees match: {trees_equal(root, decoded)}")

=== Serialize / Deserialize (#297) ===

Original tree:
1
├── 2
└── 3
    ├── 4
    └── 5

--- Serializing ---
  visit 1 → [1]
  visit 2 → [1, 2]
  null    → [1, 2, #]
  null    → [1, 2, #, #]
  visit 3 → [1, 2, #, #, 3]
  visit 4 → [1, 2, #, #, 3, 4]
  null    → [1, 2, #, #, 3, 4, #]
  null    → [1, 2, #, #, 3, 4, #, #]
  visit 5 → [1, 2, #, #, 3, 4, #, #, 5]
  null    → [1, 2, #, #, 3, 4, #, #, 5, #]
  null    → [1, 2, #, #, 3, 4, #, #, 5, #, #]

Serialized: "1,2,#,#,3,4,#,#,5,#,#"

--- Deserializing ---
  read '1' → create node(1)
  read '2' → create node(2)
  read '#' → None (left of 2)
  read '#' → None (right of 2)
  read '3' → create node(3)
  read '4' → create node(4)
  read '#' → None (left of 4)
  read '#' → None (right of 4)
  read '5' → create node(5)
  read '#' → None (left of 5)
  read '#' → None (right of 5)

Reconstructed tree:
1
├── 2
└── 3
    ├── 4
    └── 5

Trees match: True


---

## Practice Problems

| # | Problem | Pattern | Difficulty |
|---|---|---|---|
| 104 | Maximum Depth of Binary Tree | Recursive height | Easy |
| 226 | Invert Binary Tree | Recursive swap | Easy |
| 100 | Same Tree | Recursive compare | Easy |
| 101 | Symmetric Tree | Mirror check | Easy |
| 110 | Balanced Binary Tree | Height + check | Easy |
| 543 | Diameter of Binary Tree | Height + global max | Easy |
| 102 | Binary Tree Level Order Traversal | BFS with levels | Medium |
| 236 | Lowest Common Ancestor | Recursive split | Medium |
| 98 | Validate Binary Search Tree | Range passing | Medium |
| 230 | Kth Smallest Element in BST | Inorder + count | Medium |
| 108 | Convert Sorted Array to BST | Divide and conquer | Easy |
| 215 | Kth Largest Element in Array | Min-heap of size k | Medium |
| 347 | Top K Frequent Elements | Heap + frequency | Medium |
| 23 | Merge K Sorted Lists | Min-heap merge | Hard |
| 124 | Binary Tree Maximum Path Sum | DFS + global max | Hard |
| 297 | Serialize and Deserialize | Preorder + markers | Hard |
| 199 | Binary Tree Right Side View | BFS last per level | Medium |
| 105 | Construct from Preorder+Inorder | Recursive build | Medium |

---

## Tree Pattern Cheat Sheet

```
TREE PATTERN CHEAT SHEET:

"Visit all nodes"             → Choose traversal: inorder/preorder/postorder/BFS
"Height/depth"                → Recursive: max(left, right) + 1
"Check property"              → DFS returning bool or value
"Lowest common ancestor"      → Recursive: check left and right subtrees
"Sorted order from BST"       → Inorder traversal
"Kth element in BST"          → Inorder traversal, count
"Level-by-level"              → BFS with queue
"Top K / Kth largest"         → Heap (min-heap of size k)
"Path sum problems"           → DFS with running sum
"Serialize / reconstruct"     → Preorder with None markers
"Validate BST"               → Pass valid (min, max) range down
"Build BST from sorted"      → Pick middle, recurse halves
"Merge K sorted things"      → Min-heap, pop smallest, push next
```

```
TRAVERSAL QUICK REFERENCE:

  Inorder   (L, Root, R)  → BST sorted order, kth smallest
  Preorder  (Root, L, R)  → serialize, copy tree
  Postorder (L, R, Root)  → delete tree, evaluate expression
  BFS       (level by level) → shortest path, level-order problems
```

```
COMPLEXITY REFERENCE:

  Operation          │ Balanced BST │ Skewed Tree │ Heap
  ───────────────────┼──────────────┼─────────────┼──────
  Search             │   O(log n)   │    O(n)     │  —
  Insert             │   O(log n)   │    O(n)     │ O(log n)
  Delete             │   O(log n)   │    O(n)     │ O(log n)
  Get min/max        │   O(log n)   │    O(n)     │ O(1)
  Traverse all       │    O(n)      │    O(n)     │  —
```

---

**Next up: Topic 10 — Graphs** (BFS, DFS, connected components, topological sort, shortest paths)